##Load files

In [ ]:
!fusermount -u /content/drive
!rm -rf /content/drive
!mkdir /content/drive
from google.colab import drive
import numpy as np
import os

# --- Mount Google Drive ---
drive.mount('/content/drive', force_remount=True)

# --- Set path to your files ---
files_path = '/content/drive/MyDrive/CMMD_preprocessed/v1.0' #My sample dataset is the CMMD dataset from TCIA, but you can use whatever dataset that your model uses

print(f" Loading data from: {files_path}")

# --- Load Metadata & Labels ---
print(" Loading Metadata & Labels...")
y_test  = np.load(os.path.join(files_path, 'y_test.npy')) #We only need the test set here to use as ground-truth

# --- 6. VERIFICATION ---
print(f" - Labels: {y_test.shape}")
print("="*40)

##Implement Bayes-DSS

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. DATA INGESTION
# ==========================================
try:
    from google.colab import drivedaulda
    drive.mount('/content/drive')
except:
    pass

data_path = '/content/drive/MyDrive/Xception model/v1.0/results/v1.0_p2_test_predictions.npz' #You need to import your model's test set prediction result. In my case, I used Xception model but you can use your own model instead

print(f" Loading model predictions from: {data_path}")
try:
    loaded_data = np.load(data_path, allow_pickle=True)
    y_pred_p2 = loaded_data['y_pred']
    y_test = loaded_data['y_test']
    target_names = loaded_data['target_names']
    print(f" Data loaded successfully. Found predictions for {y_pred_p2.shape[0]} patients.")
except FileNotFoundError:
    print(f" Error: Could not find the file at {data_path}.")
    raise

y_true_indices = np.argmax(y_test, axis=1)
y_pred_indices = np.argmax(y_pred_p2, axis=1)
cmtx = confusion_matrix(y_true_indices, y_pred_indices)

# ==========================================
# 2. CORE MATHEMATICAL FUNCTIONS
# ==========================================
def policy_aggressive(S, R, S0, R0, **kwargs):
    # Always administers the maximum dose (1.0)
    return 1.0

def policy_adaptive(S, R, S0, R0, threshold=0.5, lower_dose=0.5, **kwargs):
    # Full dose if tumor is above threshold, otherwise drops to lower_dose (0.5)
    if (S + R) >= threshold * (S0 + R0):
        return 1.0
    return lower_dose

def policy_intermittent(S, R, S0, R0, threshold=1.0, **kwargs):
    # Full dose only if tumor returns to or exceeds threshold (1.0), otherwise 0.0
    if (S + R) >= threshold * (S0 + R0):
        return 1.0
    return 0.0

def run_tumor_simulation(params, config, pol_params):
    D = config['D']
    max_cycles = config['max_cycles']
    total_days = D * max_cycles
    results = {}

    policies = {
        'Aggressive': policy_aggressive,

        # Passes the threshold and the lower dose (0.5) from the UI
        'Adaptive': lambda S, R, S0, R0: policy_adaptive(
            S, R, S0, R0,
            threshold=pol_params['adapt_thresh'],
            lower_dose=pol_params['adapt_low_dose']
        ),

        # Passes the single threshold (1.0) from the UI
        'Intermittent': lambda S, R, S0, R0: policy_intermittent(
            S, R, S0, R0,
            threshold=pol_params['inter_thresh']
        )
    }

    for subtype, p in params.items():
        # Growth rates are already daily in the literature, no conversion needed
        r_s_daily = p['rs']
        r_r_daily = p['rr']

        # AUTOMATIC CONVERSION: Cycle Efficacy -> Daily Efficacy using the 'D' widget
        e_s_daily = 1 - (1 - p['es'])**(1/D) if p['es'] > 0 else 0
        e_r_daily = 1 - (1 - p['er'])**(1/D) if p['er'] > 0 else 0

        for pol_name, pol_func in policies.items():
            S, R = p['S0'], p['R0']
            S0, R0 = p['S0'], p['R0']

            # This directly represents \sum a_t from your thesis
            cycle_toxicity_sum = 0.0
            cycle_logs = []

            # This holds the locked dose for the duration of the cycle
            current_a_t = 0.0

            for day in range(1, total_days + 1):

                # CLINICAL REALITY CHECK: Evaluate tumor and decide dose ONLY at the start of a cycle
                if (day - 1) % D == 0:
                    current_a_t = pol_func(S, R, S0, R0)

                    # Accumulate a_t exactly ONCE per cycle, aligning with \sum_{t=1}^{T} a_t
                    cycle_toxicity_sum += current_a_t

                # Apply the dynamically calculated daily rates using the locked-in dose
                S_next = S * (1 + r_s_daily) * (1 - e_s_daily * current_a_t)
                R_next = R * (1 + r_r_daily) * (1 - e_r_daily * current_a_t)

                S, R = S_next, R_next

                # Log the snapshot of the patient at the exact boundary of the cycle
                if day % D == 0:
                    cycle_logs.append({
                        'burden': S + R,
                        # No longer dividing by D! The sum is already per-cycle
                        'sum_at': cycle_toxicity_sum
                    })

            results[(pol_name, subtype)] = cycle_logs

    return results, list(policies.keys())

def calculate_bayesian_utility(sim_results, priors_matrix, conf_matrix, config, policy_names, subtypes):
    lambdas = config['lambda_range']
    num_patients = priors_matrix.shape[0]
    num_cycles = config['max_cycles']

    col_sums = conf_matrix.sum(axis=0)
    col_sums[col_sums == 0] = 1
    p_true_given_pred = (conf_matrix / col_sums).T

    eu_grid = np.zeros((num_patients, len(policy_names), num_cycles, len(lambdas)))

    for pol_idx, pol_name in enumerate(policy_names):
        for cycle_idx in range(num_cycles):
            for lam_idx, lam in enumerate(lambdas):
                v_matrix = np.zeros(len(subtypes))
                for s_idx, subtype in enumerate(subtypes):
                    log = sim_results[(pol_name, subtype)][cycle_idx]
                    v_matrix[s_idx] = -(log['burden']) - (lam * log['sum_at'])

                for pat_idx in range(num_patients):
                    p_pred = priors_matrix[pat_idx]
                    p_true = np.dot(p_pred, p_true_given_pred)
                    eu_grid[pat_idx, pol_idx, cycle_idx, lam_idx] = np.dot(p_true, v_matrix)
    return eu_grid

def extract_all_decisions(eu_matrix, lambdas, patient_preds, patient_trues, target_names, policy_names):
    """
    Extracts the optimal cycle duration for EVERY policy, for EVERY patient, across EVERY lambda.
    Flags ALL mathematically optimal options (including ties) so doctors can compare alternatives.
    """
    num_patients = eu_matrix.shape[0]
    results_list = []

    for lam_idx, lam in enumerate(lambdas):
        for pat_idx in range(num_patients):
            # The 2D grid of (Policies x Cycles) for this specific patient and lambda
            patient_grid = eu_matrix[pat_idx, :, :, lam_idx]

            # Find the absolute highest EU score in the entire grid
            global_max_eu = np.max(patient_grid)

            # Loop through all 3 policies to provide the doctor with alternative options
            for pol_idx, pol_name in enumerate(policy_names):

                # Isolate the EU scores across all 10 cycles for THIS specific policy
                policy_scores = patient_grid[pol_idx, :]

                # Find the best cycle count for this specific alternative
                best_cycle_for_pol_idx = np.argmax(policy_scores)
                max_eu_for_pol = policy_scores[best_cycle_for_pol_idx]

               # If this alternative's EU matches the global maximum EU (handling tiny floating-point math differences), it gets a Star!
                is_absolute_best = np.isclose(max_eu_for_pol, global_max_eu, atol=1e-6)

                results_list.append({
                    'Patient_ID': f"Patient_{pat_idx+1}",
                    'True_Subtype': target_names[patient_trues[pat_idx]],
                    'Predicted_Subtype': target_names[patient_preds[pat_idx]],
                    'Lambda_Weight': round(lam, 3),
                    'Policy_Option': pol_name,
                    'Recommended_Cycles': best_cycle_for_pol_idx + 1,
                    'Expected_Utility': round(max_eu_for_pol, 4),
                    'Is_Optimal_Choice': '⭐ YES' if is_absolute_best else 'Alternative'
                })

    return pd.DataFrame(results_list)

# ==========================================
# 3. INTERACTIVE DASHBOARD UI
# ==========================================
style = {'description_width': 'initial'}

# --- UI: General Config ---
w_D = widgets.IntText(value=21, description='Days/Cycle (D):', style=style)
w_max_cycles = widgets.IntText(value=10, description='Max Cycles (n):', style=style)
w_lam_min = widgets.FloatText(value=0.01, description='Min λ:', style=style, layout=widgets.Layout(width='120px'), step=0.01)
w_lam_max = widgets.FloatText(value=0.5, description='Max λ:', style=style, layout=widgets.Layout(width='120px'), step=0.1)

# The increment step widget
w_lam_step = widgets.FloatText(value=0.01, description='Step size:', style=style, layout=widgets.Layout(width='120px'), step=0.01)

# --- UI: Policy Adjustments ---
w_adapt_thresh = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Adaptive Threshold:', style=style, layout=widgets.Layout(width='400px'))
w_adapt_low_dose = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Adaptive Low Dose:', style=style, layout=widgets.Layout(width='400px'))

w_inter_thresh = widgets.FloatSlider(value=1.0, min=0.1, max=1.5, step=0.05, description='Intermittent Threshold:', style=style, layout=widgets.Layout(width='400px'))

# --- UI: Subtype Constants ---
default_st = {
    'Benign': [1.00, 0.00, 0.00076, 0.00000, 0.00000, 0.00000],
    'LumA':   [0.07, 0.03, 0.0027, 0.00135, 0.20, 0.0325],
    'LumB':   [0.90, 0.10, 0.0033, 0.00165, 0.35, 0.0055],
    'HER2':   [0.825, 0.175, 0.0038, 0.00304, 0.60, 0.125],
    'TN':     [0.60, 0.40, 0.0055, 0.0044, 0.55, 0.085]
}

st_widgets = {}
accordions = []
for st_name, vals in default_st.items():
    s0 = widgets.FloatText(value=vals[0], description='S0:', layout=widgets.Layout(width='140px'), step=0.01)
    r0 = widgets.FloatText(value=vals[1], description='R0:', layout=widgets.Layout(width='140px'), step=0.01)
    rs = widgets.FloatText(value=vals[2], description='rs (daily):', layout=widgets.Layout(width='140px'), step=0.00001)
    rr = widgets.FloatText(value=vals[3], description='rr (daily):', layout=widgets.Layout(width='140px'), step=0.00001)
    es = widgets.FloatText(value=vals[4], description='es (per cycle):', layout=widgets.Layout(width='140px'), step=0.00001)
    er = widgets.FloatText(value=vals[5], description='er (per cycle):', layout=widgets.Layout(width='140px'), step=0.00001)

    st_widgets[st_name] = {'S0': s0, 'R0': r0, 'rs': rs, 'rr': rr, 'es': es, 'er': er}
    row1 = widgets.HBox([s0, r0])
    row2 = widgets.HBox([rs, rr])
    row3 = widgets.HBox([es, er])
    accordions.append(widgets.VBox([row1, row2, row3]))

accordion_ui = widgets.Accordion(children=accordions)
for i, st_name in enumerate(default_st.keys()):
    accordion_ui.set_title(i, f"Constants: {st_name}")

# --- UI: Execution ---
btn_run = widgets.Button(description='Run Simulation & Extract All', button_style='success', layout=widgets.Layout(width='250px', height='40px'))
out_console = widgets.Output()

# Organize layout
ui_config = widgets.VBox([
    widgets.HTML("<b>1. Simulation Config</b>"),
    widgets.HBox([w_D, w_max_cycles]),
    widgets.HBox([w_lam_min, w_lam_max, w_lam_step]),
    widgets.HTML("<hr><b>2. Treatment Policies (a_t)</b>"),
    w_adapt_thresh, w_adapt_low_dose, w_inter_thresh,
    widgets.HTML("<hr><b>3. Biological Constants</b>"),
    accordion_ui,
    widgets.HTML("<br>"),
    btn_run
])

display(ui_config, out_console)

# ==========================================
# 4. EXECUTION LOGIC (Triggered by Button)
# ==========================================
def on_btn_click(b):
    with out_console:
        clear_output()
        print(" Gathering parameters from dashboard...")

        # Create the array using the explicit step size.
        # Adding a tiny fraction to the max ensures the final cap is included in the list.
        lam_array = np.arange(w_lam_min.value, w_lam_max.value + (w_lam_step.value / 2), w_lam_step.value)

        sim_config = {
            'D': w_D.value,
            'max_cycles': w_max_cycles.value,
            'lambda_range': lam_array
        }

        pol_params = {
            'adapt_thresh': w_adapt_thresh.value,
            'adapt_low_dose': w_adapt_low_dose.value,
            'inter_thresh': w_inter_thresh.value
        }

        subtype_params = {}
        for st_name, ws in st_widgets.items():
            subtype_params[st_name] = {k: v.value for k, v in ws.items()}

        print(" Running Daily Tumor Simulation Matrix...")
        sim_results, policy_names = run_tumor_simulation(subtype_params, sim_config, pol_params)

        print(" Calculating Bayesian Expected Utility Grid...")
        eu_matrix = calculate_bayesian_utility(
            sim_results, y_pred_p2, cmtx, sim_config, policy_names, list(subtype_params.keys())
        )

        print(f" Extracting full dataset across {len(lam_array)} λ values...")
        df_results = extract_all_decisions(
            eu_matrix, sim_config['lambda_range'],
            y_pred_indices, y_true_indices, target_names, policy_names
        )

        csv_path = '/content/drive/MyDrive/Xception model/v1.0/treatments/optimal_treatments_full_grid.csv'
        os.makedirs(os.path.dirname(csv_path), exist_ok=True)
        df_results.to_csv(csv_path, index=False)

        print(f" DONE! Generated {len(df_results)} recommendation rows.")
        print(f" Full dataset saved to: {csv_path}")

        print("\n Snapshot of Recommendations (Sorted by Patient):")

        # Clean numeric sort so Patient_1 is followed by Patient_2
        df_results['Temp_Num'] = df_results['Patient_ID'].str.extract(r'(\d+)').astype(int)
        df_results = df_results.sort_values(
            by=['Temp_Num', 'Lambda_Weight', 'Expected_Utility'],
            ascending=[True, True, False]
        ).drop(columns=['Temp_Num'])

        display(df_results.head(30))

btn_run.on_click(on_btn_click)

###Case by case analysis

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. LOAD THE SAVED DATA
# ==========================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

csv_path = '/content/drive/MyDrive/Xception model/v1.0/treatments/optimal_treatments_full_grid.csv'

try:
    print(f" Loading results from: {csv_path}")
    df_results = pd.read_csv(csv_path)
    print(f" Successfully loaded {len(df_results)} rows of clinical recommendations.")
except FileNotFoundError:
    print(f" Error: Could not find the file at {csv_path}. Did the main simulation finish running?")
    raise

# --- NATURAL SORTING ---
# 1. Extract the number from the 'Patient_ID' string and temporarily save it as a real integer
df_results['Temp_Sort_Num'] = df_results['Patient_ID'].str.extract(r'(\d+)').astype(int)

# 2. Sort using this real integer, then by Lambda, then by EU
df_results = df_results.sort_values(
    by=['Temp_Sort_Num', 'Lambda_Weight', 'Expected_Utility'],
    ascending=[True, True, False]
)

# 3. Delete the temporary integer column to keep the table clean
df_results = df_results.drop(columns=['Temp_Sort_Num'])

# ==========================================
# 2. INTERACTIVE VIEWER UI
# ==========================================
# Create a list of all unique patients in the CSV
patient_list = df_results['Patient_ID'].unique()

# The Dropdown Widget
w_patient = widgets.Dropdown(
    options=patient_list,
    value=patient_list[0], # Defaults to the first patient
    description='Select Case:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

# A toggle to hide the inferior alternatives and only show the "⭐ YES" optimal choices
w_optimal_only = widgets.Checkbox(
    value=False,
    description='Show mathematically optimal choices only',
    style={'description_width': 'initial'}
)

out_table = widgets.Output(layout=widgets.Layout(width='100%', overflow='auto'))

def update_table(*args):
    """Filters the dataframe based on the dropdown and checkbox, then displays it."""
    with out_table:
        clear_output()

        # 1. Filter by the selected patient
        filtered_df = df_results[df_results['Patient_ID'] == w_patient.value]

        # 2. Filter out alternative options if the checkbox is checked
        if w_optimal_only.value:
            filtered_df = filtered_df[filtered_df['Is_Optimal_Choice'].str.contains('YES', na=False)]

        # Force pandas to show all rows AND all columns for the selected patient
        with pd.option_context('display.max_rows', None, 'display.max_columns', None):
            display(filtered_df)

# Attach the update function to the widgets so the table changes instantly when clicked
w_patient.observe(update_table, 'value')
w_optimal_only.observe(update_table, 'value')

# Build the visual layout
ui = widgets.VBox([
    widgets.HTML("<h3>📊 Clinical Decision Support Viewer</h3>"),
    widgets.HBox([w_patient, w_optimal_only]),
    out_table
])

# Display the UI and trigger the first view
display(ui)
update_table()

###Overall analysis

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# ==========================================
# 1. LOAD & PREP DATA
# ==========================================
csv_path = '/content/drive/MyDrive/Xception model/v1.0/treatments/optimal_treatments_full_grid.csv'

try:
    df_full = pd.read_csv(csv_path)

    # We must isolate ONLY the mathematically optimal choices
    # This strips away all the inferior "Alternative" rows so they don't corrupt our averages
    df_optimal = df_full[df_full['Is_Optimal_Choice'] == '⭐ YES'].copy()

    print(f" Successfully loaded data: {len(df_full)} total rows.")
    print(f" Filtered to {len(df_optimal)} mathematically optimal decisions for analysis.")
except FileNotFoundError:
    print(f" File not found at: {csv_path}")
    raise

# ==========================================
# 2. OVERALL SUMMARY
# ==========================================
def print_overall_summary(data):
    print("\n" + "="*50)
    print(" OVERALL POLICY DISTRIBUTION (OPTIMAL CHOICES)")
    print("="*50)

    # Count of each policy (Updated to use the new 'Policy_Option' column name)
    policy_counts = data['Policy_Option'].value_counts()
    for policy, count in policy_counts.items():
        percentage = (count / len(data)) * 100
        print(f" - {policy}: {count} cases ({percentage:.1f}%)")

    print("\n AVERAGE CYCLES PER POLICY")
    avg_cycles = data.groupby('Policy_Option')['Recommended_Cycles'].mean().round(2)
    for policy, cycles in avg_cycles.items():
        print(f" - {policy}: {cycles} cycles")

# ==========================================
# 3. SUBTYPE ANALYSIS
# ==========================================
def analyze_by_subtype(data):
    print("\n" + "="*50)
    print(" TRUE SUBTYPE ANALYSIS")
    print("="*50)

    # Cross-tabulation between True Subtype and Policy_Option
    crosstab = pd.crosstab(data['True_Subtype'], data['Policy_Option'], margins=True, margins_name="Total")
    print(crosstab)
    print("\n")

    # Average cycles per true subtype
    print("Average Recommended Cycles by True Subtype:")
    print(data.groupby('True_Subtype')['Recommended_Cycles'].mean().round(2))

# ==========================================
# 4. DIAGNOSTIC UNCERTAINTY (BAYESIAN IMPACT)
# ==========================================
def analyze_diagnostic_mismatches(data):
    """
    Examines cases where the Xception model's prediction was incorrect,
    but the Bayesian Expected Utility still had to make a decision.
    """
    print("\n" + "="*50)
    print(" DIAGNOSTIC MISMATCH ANALYSIS")
    print("="*50)

    # Filter incorrect predictions
    mismatches = data[data['True_Subtype'] != data['Predicted_Subtype']]
    match_rate = 100 - ((len(mismatches) / len(data)) * 100)

    print(f"Total optimal decisions in mismatched cases: {len(mismatches)} / {len(data)} (Overall Accuracy: {match_rate:.1f}%)")

    if len(mismatches) > 0:
        print("\nPolicy distribution in mismatched cases:")
        print(mismatches['Policy_Option'].value_counts())

        print("\nSample of up to 5 random mismatches:")
        display(mismatches.sample(min(5, len(mismatches))))

# ==========================================
# 5. DEEP DIVE SEARCH
# ==========================================
def search_specific_cases(data, true_subtype=None, policy=None, min_cycles=None):
    """
    Function to filter and examine highly specific subsets of patients.
    """
    filtered_df = data.copy()

    if true_subtype:
        filtered_df = filtered_df[filtered_df['True_Subtype'] == true_subtype]
    if policy:
        filtered_df = filtered_df[filtered_df['Policy_Option'] == policy]
    if min_cycles is not None:
        filtered_df = filtered_df[filtered_df['Recommended_Cycles'] >= min_cycles]

    print(f"\n DEEP DIVE SEARCH: {len(filtered_df)} results found")
    if len(filtered_df) > 0:
        display(filtered_df.head(10)) # Show top 10
    return filtered_df

# ==========================================
# 6. EXECUTION
# ==========================================
# Call the analysis functions using the FILTERED dataframe
print_overall_summary(df_optimal)
analyze_by_subtype(df_optimal)
analyze_diagnostic_mismatches(df_optimal)

##Sensitivity Analysis

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. LOAD PREDICTIONS & MATRICES
# ==========================================
data_path = '/content/drive/MyDrive/Xception model/v1.0/results/v1.0_p2_test_predictions.npz'
try:
    loaded_data = np.load(data_path, allow_pickle=True)
    y_pred_p2 = loaded_data['y_pred']
    y_test = loaded_data['y_test']
    target_names = list(loaded_data['target_names'])
    y_true_indices = np.argmax(y_test, axis=1)
    y_pred_indices = np.argmax(y_pred_p2, axis=1)
    cmtx = confusion_matrix(y_true_indices, y_pred_indices)
except FileNotFoundError:
    print(f" Error: Could not find the file at {data_path}.")
    raise

# ==========================================
# 2. CORE MATHEMATICAL ENGINE
# ==========================================
def policy_aggressive(S, R, S0, R0): return 1.0
def policy_adaptive(S, R, S0, R0, threshold=0.5, lower_dose=0.5): return 1.0 if (S + R) >= threshold * (S0 + R0) else lower_dose
def policy_intermittent(S, R, S0, R0, threshold=1.0): return 1.0 if (S + R) >= threshold * (S0 + R0) else 0.0

def run_tumor_simulation(params, config, pol_params):
    D = config['D']
    total_days = D * config['max_cycles']
    results = {}
    policies = {
        'Aggressive': policy_aggressive,
        'Adaptive': lambda S, R, S0, R0: policy_adaptive(S, R, S0, R0, pol_params['adapt_thresh'], pol_params['adapt_low_dose']),
        'Intermittent': lambda S, R, S0, R0: policy_intermittent(S, R, S0, R0, pol_params['inter_thresh'])
    }

    for subtype, p in params.items():
        r_s_daily, r_r_daily = p['rs'], p['rr']
        e_s_daily = 1 - (1 - p['es'])**(1/D) if p['es'] > 0 else 0
        e_r_daily = 1 - (1 - p['er'])**(1/D) if p['er'] > 0 else 0

        for pol_name, pol_func in policies.items():
            S, R, S0, R0 = p['S0'], p['R0'], p['S0'], p['R0']
            cycle_toxicity_sum, current_a_t = 0.0, 0.0
            cycle_logs = []

            for day in range(1, total_days + 1):
                if (day - 1) % D == 0:
                    current_a_t = pol_func(S, R, S0, R0)
                    cycle_toxicity_sum += current_a_t

                S = S * (1 + r_s_daily) * (1 - e_s_daily * current_a_t)
                R = R * (1 + r_r_daily) * (1 - e_r_daily * current_a_t)

                if day % D == 0:
                    cycle_logs.append({'burden': S + R, 'sum_at': cycle_toxicity_sum})
            results[(pol_name, subtype)] = cycle_logs
    return results, list(policies.keys())

def calculate_bayesian_utility(sim_results, priors_matrix, conf_matrix, config, policy_names, subtypes):
    lambdas = config['lambda_range']
    col_sums = conf_matrix.sum(axis=0)
    col_sums[col_sums == 0] = 1
    p_true_given_pred = (conf_matrix / col_sums).T
    eu_grid = np.zeros((priors_matrix.shape[0], len(policy_names), config['max_cycles'], len(lambdas)))

    for pol_idx, pol_name in enumerate(policy_names):
        for cycle_idx in range(config['max_cycles']):
            for lam_idx, lam in enumerate(lambdas):
                v_matrix = np.zeros(len(subtypes))
                for s_idx, subtype in enumerate(subtypes):
                    log = sim_results[(pol_name, subtype)][cycle_idx]
                    v_matrix[s_idx] = -(log['burden']) - (lam * log['sum_at'])

                for pat_idx in range(priors_matrix.shape[0]):
                    p_true = np.dot(priors_matrix[pat_idx], p_true_given_pred)
                    eu_grid[pat_idx, pol_idx, cycle_idx, lam_idx] = np.dot(p_true, v_matrix)
    return eu_grid

# ==========================================
# 3. INTERACTIVE SENSITIVITY DASHBOARD
# ==========================================
style = {'description_width': 'initial'}

# --- UI: Shock Configuration ---
w_subtype = widgets.Dropdown(options=target_names, value='TN', description='Target Subtype:', style=style)

# NEW: The Specific Treatment Dropdown
w_policy = widgets.Dropdown(
    options=['All Optimal (Global Best)', 'Aggressive', 'Adaptive', 'Intermittent'],
    value='All Optimal (Global Best)',
    description='Target Treatment:',
    style=style, layout=widgets.Layout(width='300px')
)

w_shock_var = widgets.Dropdown(
    options=[('Sensitive Initial Size (S0)', 'S0'), ('Resistant Initial Size (R0)', 'R0'),
             ('Sensitive Growth Rate (rs)', 'rs'), ('Resistant Growth Rate (rr)', 'rr'),
             ('Sensitive Drug Efficacy (es)', 'es'), ('Resistant Drug Efficacy (er)', 'er')],
    value='es', description='Variable to Shock:', style=style, layout=widgets.Layout(width='300px')
)
w_shock_min = widgets.FloatText(value=-0.3, description='Min Shock (e.g. -0.3):', style=style, layout=widgets.Layout(width='180px'), step=0.05)
w_shock_max = widgets.FloatText(value=0.3, description='Max Shock (e.g. 0.3):', style=style, layout=widgets.Layout(width='180px'), step=0.05)
w_shock_step = widgets.FloatText(value=0.1, description='Increment Step:', style=style, layout=widgets.Layout(width='180px'), step=0.01)

# --- UI: Static Simulation Constants ---
w_lambda = widgets.FloatText(value=0.10, description='Static \u03bb Weight:', style=style, layout=widgets.Layout(width='180px'), step=0.01)
w_D = widgets.IntText(value=21, description='Days/Cycle (D):', style=style, layout=widgets.Layout(width='180px'))
w_max_cycles = widgets.IntText(value=10, description='Max Cycles (n):', style=style, layout=widgets.Layout(width='180px'))

w_adapt_thresh = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description='Adaptive Thresh:', style=style)
w_adapt_low_dose = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05, description='Adaptive Low Dose:', style=style)
w_inter_thresh = widgets.FloatSlider(value=1.0, min=0.1, max=1.5, step=0.05, description='Intermittent Thresh:', style=style)

btn_run_sens = widgets.Button(description='Run Sensitivity Analysis', button_style='primary', layout=widgets.Layout(width='250px', height='40px'))
out_sens = widgets.Output()

ui_sens = widgets.VBox([
    widgets.HTML("<h3>🔬 Comprehensive Sensitivity Analysis Dashboard</h3>"),
    widgets.HTML("<b>1. Shock Configuration</b> (What are we testing?)"),
    widgets.HBox([w_subtype, w_policy, w_shock_var]), # Added policy dropdown here
    widgets.HBox([w_shock_min, w_shock_max, w_shock_step]),
    widgets.HTML("<hr><b>2. Static Parameters</b> (What stays constant?)"),
    widgets.HBox([w_lambda, w_D, w_max_cycles]),
    widgets.HBox([w_adapt_thresh, w_adapt_low_dose, w_inter_thresh]),
    widgets.HTML("<br>"),
    btn_run_sens,
    out_sens
])

display(ui_sens)

# ==========================================
# 4. EXECUTION LOGIC
# ==========================================
var_map = {'S0': 0, 'R0': 1, 'rs': 2, 'rr': 3, 'es': 4, 'er': 5}

def on_run_sens_click(b):
    with out_sens:
        clear_output()
        print(" Running permutations. Please wait...")

        default_st = {
            'Benign': [1.00, 0.00, 0.00076, 0.00000, 0.00000, 0.00000],
            'LumA':   [0.07, 0.03, 0.0027, 0.00135, 0.20, 0.0325],
            'LumB':   [0.90, 0.10, 0.0033, 0.00165, 0.35, 0.0055],
            'HER2':   [0.825, 0.175, 0.0038, 0.00304, 0.60, 0.125],
            'TN':     [0.60, 0.40, 0.0055, 0.0044, 0.55, 0.085]
        }

        target_st = w_subtype.value
        target_idx = target_names.index(target_st)
        var_to_shock = w_shock_var.value
        v_idx = var_map[var_to_shock]
        target_policy = w_policy.value

        shock_levels = np.arange(w_shock_min.value, w_shock_max.value + (w_shock_step.value / 2), w_shock_step.value)

        sim_config = {'D': w_D.value, 'max_cycles': w_max_cycles.value, 'lambda_range': [w_lambda.value]}
        pol_params = {'adapt_thresh': w_adapt_thresh.value, 'adapt_low_dose': w_adapt_low_dose.value, 'inter_thresh': w_inter_thresh.value}

        tn_patient_indices = np.where(y_true_indices == target_idx)[0]
        if len(tn_patient_indices) == 0:
            print(f" Warning: No patients found with true subtype {target_st} in the test set.")
            return

        sensitivity_results = []

        for shock in shock_levels:
            shocked_params = copy.deepcopy(default_st)
            base_val = shocked_params[target_st][v_idx]

            new_val = base_val * (1 + shock)
            shocked_params[target_st][v_idx] = new_val

            sim_params = {k: {'S0':v[0], 'R0':v[1], 'rs':v[2], 'rr':v[3], 'es':v[4], 'er':v[5]} for k, v in shocked_params.items()}

            sim_results, policy_names = run_tumor_simulation(sim_params, sim_config, pol_params)
            eu_matrix = calculate_bayesian_utility(sim_results, y_pred_p2, cmtx, sim_config, policy_names, list(sim_params.keys()))

            cycle_counts = []
            for pat_idx in tn_patient_indices:
                pat_grid = eu_matrix[pat_idx, :, :, 0]

                # Isolate the specific policy or find the global best
                if target_policy == 'All Optimal (Global Best)':
                    global_max_eu = np.max(pat_grid)
                    best_indices = np.where(np.isclose(pat_grid, global_max_eu, atol=1e-6))
                    best_cycle = best_indices[1][0] + 1
                else:
                    pol_idx = policy_names.index(target_policy)
                    policy_scores = pat_grid[pol_idx, :]
                    best_cycle = np.argmax(policy_scores) + 1

                cycle_counts.append(best_cycle)

            avg_cycles = np.mean(cycle_counts)

            sensitivity_results.append({
                'Shock Variance': f"{shock*100:+.1f}%",
                f'Modified {var_to_shock} ({target_st})': round(new_val, 5),
                'Avg Recommended Cycles': round(avg_cycles, 2)
            })

        df_sens = pd.DataFrame(sensitivity_results)

        clear_output()
        print(f" Analysis Complete! Target: {target_st} | Variable: {var_to_shock} | Policy: {target_policy} | \u03bb: {w_lambda.value}")
        display(df_sens)

        plt.figure(figsize=(9, 5))
        plt.plot(shock_levels * 100, df_sens['Avg Recommended Cycles'], marker='o', linestyle='-', color='#d62728', linewidth=2)
        plt.title(f'Sensitivity Analysis: {w_shock_var.label} vs Treatment Duration\n(Subtype: {target_st}, Policy: {target_policy}, $\lambda={w_lambda.value}$)', fontsize=12)
        plt.xlabel(f'Variance from Baseline {var_to_shock} (%)', fontsize=10)
        plt.ylabel('Average Recommended Cycles', fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(shock_levels * 100, [f"{s*100:+.0f}%" for s in shock_levels])
        plt.tight_layout()
        plt.show()

btn_run_sens.on_click(on_run_sens_click)